In [ ]:
import os
import re
from pathlib import Path
from string import ascii_lowercase

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv

for env_path in (Path(".env"), Path("../.env"), Path("machine-learning/.env")):
    load_dotenv(env_path)

if "MPL_STYLE" not in os.environ:
    os.environ["MPL_STYLE"] = "seaborn-v0_8-notebook"
plt.style.use(os.environ["MPL_STYLE"])

OUTPUT_BASE = Path(os.getenv("OUTPUT_BASE")).resolve(strict=True)


In [ ]:
DATASET_LABEL = "M6"
EXPERIMENT_DIR = OUTPUT_BASE / "experiments" / DATASET_LABEL
SUMMARY_DIR = OUTPUT_BASE / "summary" / DATASET_LABEL
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
PARQUET_COLUMNS = ["snapshot_id", "time", "prediction_physical", "target_physical"]

RESIDUAL_CONFIGS = [
    {
        "experiment_name": "from+x+y+z+vx+vy+vz-to+time-nstar128-nsnap16-dp0p6-dr0p1_0p9-set_transformer-hd6-nh2-ns2-no_ohd-d0p1-bs10240-lr0p0001-wd0p001",
    },
    {
        "experiment_name": "from+x+y+z+vx+vy+vz+log_L_L_sol-to+total_mass_within_2x_r_tidal-nstar128-nsnap16-dp0p6-dr0p1_0p9-summary_stats-h12-d0p1-bs10240-lr0p0001-wd0p001",
    },
]


In [ ]:
FEATURE_SET_LABELS = {
    "lon_deg+lat_deg+pm_lon_coslat_mas_yr+pm_lat_mas_yr": "Sky",
    "lon_deg+lat_deg+pm_lon_coslat_mas_yr+pm_lat_mas_yr+log_L_L_sol": "Sky$+L$",
    "x+y+z+vx+vy+vz": "Cartesian",
    "x+y+z+vx+vy+vz+log_L_L_sol": "Cartesian$+L$",
}

TARGET_LABELS = {
    "time": {"ylabel": r"$\Delta t\;[\mathrm{Myr}]$"},
    "total_mass_within_2x_r_tidal": {
        "ylabel": r"$\Delta M_{\mathrm{total}}\;[M_\odot]$",
    },
}

TIME_BIN_WIDTH = 5
TIME_RANGE = (0, 320)
LOG_TIME_RANGE = (TIME_BIN_WIDTH / 2, 400)

MODEL_FAMILY_CONFIG = {
    "summary_stats": {"label_short": "SS", "color": "#1f77b4"},
    "deep_sets": {"label_short": "DS", "color": "#d62728"},
    "set_transformer": {"label_short": "ST", "color": "#2ca02c"},
}

FILENAME_PATTERN = re.compile(
    r"from\+(?P<feature_set_key>.+)-to\+(?P<target_key>.+)-nstar"
    r".*-(?P<model_family>summary_stats|deep_sets|set_transformer)-(?P<model_hparams>.+?)-"
    r"bs\d+.*$"
)


In [ ]:
# Build binned snapshot-level residual summaries once.
time_edges = np.arange(TIME_RANGE[0], TIME_RANGE[1] + TIME_BIN_WIDTH, TIME_BIN_WIDTH)
time_centers = time_edges[:-1] + 0.5 * TIME_BIN_WIDTH
plot_configs = []

for cfg in RESIDUAL_CONFIGS:
    experiment_name = cfg["experiment_name"]
    parquet_path = (
        EXPERIMENT_DIR
        / experiment_name
        / f"{experiment_name}-{DATASET_LABEL}-seed42-test.parquet"
    ).resolve(strict=True)
    match = FILENAME_PATTERN.search(experiment_name)

    feature_set_key = match.group("feature_set_key")
    target_key = match.group("target_key")
    model_family = match.group("model_family")
    model_hparams = match.group("model_hparams")

    if model_family == "summary_stats":
        model_label = f"SS[{re.search(r'h(\d+)', model_hparams).group(1)}]"
    elif model_family == "deep_sets":
        model_match = re.search(r"phi(\d+)-rho(\d+)", model_hparams)
        model_label = f"DS[{model_match.group(1)},{model_match.group(2)}]"
    else:
        model_match = re.search(r"hd(\d+)-nh(\d+)-ns(\d+)", model_hparams)
        model_label = (
            f"ST[{model_match.group(1)},{model_match.group(2)},{model_match.group(3)}]"
        )

    residual_df = pd.read_parquet(parquet_path, columns=PARQUET_COLUMNS)
    snapshot_df = (
        residual_df.groupby(["snapshot_id", "time"], sort=False, observed=True)
        .agg(
            delta=("prediction_physical", "mean"),
            target_physical=("target_physical", "first"),
        )
        .reset_index()
    )
    snapshot_df["delta"] -= snapshot_df["target_physical"]
    del residual_df

    snapshot_df["time_bin"] = pd.cut(
        snapshot_df["time"],
        bins=time_edges,
        labels=time_centers,
        include_lowest=True,
    )

    plot_df = (
        snapshot_df.dropna(subset=["time_bin"])
        .groupby("time_bin", observed=True)["delta"]
        .agg(
            q5=lambda x: x.quantile(0.05),
            q16=lambda x: x.quantile(0.16),
            median="median",
            q84=lambda x: x.quantile(0.84),
            q95=lambda x: x.quantile(0.95),
        )
        .reset_index()
    )
    plot_df["time_bin"] = plot_df["time_bin"].astype(float)

    plot_configs.append(
        {
            "target_key": target_key,
            "plot_df": plot_df,
            "color": MODEL_FAMILY_CONFIG[model_family]["color"],
            "label": f"{model_label}, {FEATURE_SET_LABELS[feature_set_key]}",
        }
    )


In [ ]:
# Plain residual scale.
fig, (age_ax, mass_ax) = plt.subplots(1, 2, figsize=(12, 5), dpi=300)
target_axes = {"time": age_ax, "total_mass_within_2x_r_tidal": mass_ax}

for plot_cfg in plot_configs:
    ax = target_axes[plot_cfg["target_key"]]
    plot_df = plot_cfg["plot_df"]
    color = plot_cfg["color"]

    band = ax.fill_between(
        plot_df["time_bin"],
        plot_df["q16"],
        plot_df["q84"],
        color=color,
        lw=0,
        alpha=0.18,
    )
    median_line = ax.plot(plot_df["time_bin"], plot_df["median"], color=color, lw=2.5)[0]
    lower_line = ax.plot(plot_df["time_bin"], plot_df["q5"], color=color, lw=1.4, ls=":", alpha=0.85)[0]
    upper_line = ax.plot(plot_df["time_bin"], plot_df["q95"], color=color, lw=1.4, ls="--", alpha=0.85)[0]

    ax.legend(
        [median_line, band, lower_line, upper_line],
        ["Median", "16-84%", "5%", "95%"],
        title=plot_cfg["label"],
        frameon=True,
        loc="lower center",
        bbox_to_anchor=(0.5, 1.02),
        fontsize=14,
        title_fontsize=14,
        ncol=2,
    )

for ax_idx, (target_key, ax) in enumerate(target_axes.items()):
    ax.text(
        0.96,
        0.96,
        f"({ascii_lowercase[ax_idx]})",
        transform=ax.transAxes,
        fontsize=22,
        va="top",
        ha="right",
    )
    ax.set_xlabel(r"True $t\;[\mathrm{Myr}]$")
    ax.set_ylabel(TARGET_LABELS[target_key]["ylabel"])
    ax.set_xlim(TIME_RANGE)
    ax.grid(True, color="grey", which="major", axis="both", ls=":", lw=0.8, alpha=0.6)
    ax.axhline(0, color="black", ls="--", lw=1, alpha=0.8)

fig.subplots_adjust(top=0.72, wspace=0.3)
fig.savefig(SUMMARY_DIR / f"{DATASET_LABEL}-residual.pdf", bbox_inches="tight")


In [ ]:
# Log target-time scale.
fig, (age_ax, mass_ax) = plt.subplots(1, 2, figsize=(12, 5), dpi=300)
target_axes = {"time": age_ax, "total_mass_within_2x_r_tidal": mass_ax}

for plot_cfg in plot_configs:
    ax = target_axes[plot_cfg["target_key"]]
    plot_df = plot_cfg["plot_df"]
    color = plot_cfg["color"]

    band = ax.fill_between(
        plot_df["time_bin"],
        plot_df["q16"],
        plot_df["q84"],
        color=color,
        lw=0,
        alpha=0.18,
    )
    median_line = ax.plot(plot_df["time_bin"], plot_df["median"], color=color, lw=2.5)[0]
    lower_line = ax.plot(plot_df["time_bin"], plot_df["q5"], color=color, lw=1.4, ls=":", alpha=0.85)[0]
    upper_line = ax.plot(plot_df["time_bin"], plot_df["q95"], color=color, lw=1.4, ls="--", alpha=0.85)[0]

    ax.legend(
        [median_line, band, lower_line, upper_line],
        ["Median", "16-84%", "5%", "95%"],
        title=plot_cfg["label"],
        frameon=True,
        loc="lower center",
        bbox_to_anchor=(0.5, 1.02),
        fontsize=14,
        title_fontsize=14,
        ncol=2,
    )

for ax_idx, (target_key, ax) in enumerate(target_axes.items()):
    ax.text(
        0.96,
        0.96,
        f"({ascii_lowercase[ax_idx]})",
        transform=ax.transAxes,
        fontsize=22,
        va="top",
        ha="right",
    )
    ax.set_xlabel(r"True $t\;[\mathrm{Myr}]$")
    ax.set_ylabel(TARGET_LABELS[target_key]["ylabel"])
    ax.set_xscale("log")
    ax.set_xlim(LOG_TIME_RANGE)
    ax.grid(True, color="grey", which="major", axis="both", ls=":", lw=0.8, alpha=0.6)
    ax.axhline(0, color="black", ls="--", lw=1, alpha=0.8)

fig.subplots_adjust(top=0.72, wspace=0.3)
fig.savefig(SUMMARY_DIR / f"{DATASET_LABEL}-residual-log.pdf", bbox_inches="tight")
